# MAINTAIN AI — Predictive Maintenance Bootstrap Training

This notebook is the reproducible Google Colab entry point for bootstrap training from real public PHM data. It automatically finds and extracts the NASA C-MAPSS #6 archive, processes **FD001–FD004**, creates leakage-safe temporal sequences, trains the shared category-aware model, evaluates against the official C-MAPSS test targets, and exports the model bundle.

**Important:** C-MAPSS is a turbofan-engine simulation dataset. It is used here for temporal degradation/RUL pretraining, not as compressor/motor data. The production MAINTAIN AI 24h/48h/7d failure-risk labels will later come from real MAINTAIN AI timestamps and technician-confirmed outcomes.

In [ ]:
# 1. Clone the Lab branch and install dependencies
!rm -rf /content/Maintain.ai.3
!git clone -b Lab https://github.com/jadhavdurvesh/Maintain.ai.3.git /content/Maintain.ai.3
%cd /content/Maintain.ai.3
!git branch --show-current
!pip install -q -r training/requirements.txt

In [ ]:
# 2. Check the actual Colab GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected. In Colab choose Runtime -> Change runtime type -> GPU, then rerun this cell.')

## 3. Upload the NASA C-MAPSS archive

Upload **only NASA dataset #6 — Turbofan Engine Degradation Simulation Data Set.zip**. Put the ZIP directly in `/content/training-data/`. You do **not** need to manually upload FD001/FD002/FD003/FD004 separately.

The next cell automatically extracts the archive and locates all four subsets plus their official RUL files.

In [ ]:
from pathlib import Path
import zipfile, shutil, glob
DATA=Path('/content/training-data')
RAW=DATA/'cmapss'
PREP=Path('/content/prepared/cmapss')
SPLIT=Path('/content/splits/cmapss')
SEQ=Path('/content/sequences')
ART=Path('/content/artifacts')
for p in (DATA,RAW,PREP,SPLIT,SEQ,ART): p.mkdir(parents=True,exist_ok=True)

zips=sorted(DATA.glob('*.zip'))
if not zips:
    raise FileNotFoundError('Upload the NASA #6 C-MAPSS ZIP into /content/training-data/ first.')
if len(zips)>1:
    print('ZIP files found:', [z.name for z in zips])
zip_path=zips[0]
print('Using:', zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(RAW)

def find_file(name):
    matches=list(RAW.rglob(name))
    if not matches:
        raise FileNotFoundError(f'Could not find {name} after extracting {zip_path.name}')
    return matches[0]

files={}
for subset in ('FD001','FD002','FD003','FD004'):
    files[subset]={
        'train':find_file(f'train_{subset}.txt'),
        'test':find_file(f'test_{subset}.txt'),
        'rul':find_file(f'RUL_{subset}.txt')}
    print(subset, files[subset])

In [ ]:
# 4. Convert ALL C-MAPSS subsets automatically
import subprocess, sys
train_parts=[]
test_parts=[]
for subset,paths in files.items():
    train_out=PREP/f'train_{subset}.parquet'
    test_out=PREP/f'test_{subset}.parquet'
    subprocess.run([sys.executable,'-m','adapters.build_common','--dataset','cmapss','--path',str(paths['train']),'--out',str(train_out)],check=True)
    subprocess.run([sys.executable,'-m','adapters.build_common','--dataset','cmapss','--path',str(paths['test']),'--rul-path',str(paths['rul']),'--out',str(test_out)],check=True)
    train_parts.append(__import__('pandas').read_parquet(train_out))
    test_parts.append(__import__('pandas').read_parquet(test_out))

import pandas as pd
train_df=pd.concat(train_parts,ignore_index=True)
test_df=pd.concat(test_parts,ignore_index=True)
train_all=PREP/'train_all.parquet'
test_all=PREP/'test_all.parquet'
train_df.to_parquet(train_all,index=False)
test_df.to_parquet(test_all,index=False)
print(f'TRAIN rows={len(train_df):,} assets={train_df.asset_id.nunique():,}')
print(f'TEST rows={len(test_df):,} assets={test_df.asset_id.nunique():,}')
print('Subsets:', sorted(train_df.subset.unique()))

In [ ]:
# 5. Inspect the normalized data
print(train_df.groupby('subset').agg(rows=('asset_id','size'),assets=('asset_id','nunique')))
display(train_df.head())
display(test_df.groupby('subset').agg(rows=('asset_id','size'),assets=('asset_id','nunique')))

In [ ]:
# 6. Split TRAIN assets only. The official C-MAPSS test engines remain untouched.
!PYTHONPATH=training/src python training/src/split_assets.py \
  --input /content/prepared/cmapss/train_all.parquet \
  --out-dir /content/splits/cmapss

In [ ]:
# 7. Build temporal sequences
!PYTHONPATH=training/src python training/src/build_sequences.py \
  --input /content/splits/cmapss/train.parquet \
  --out /content/sequences/cmapss_train.pt \
  --sequence-length 24

!PYTHONPATH=training/src python training/src/build_sequences.py \
  --input /content/prepared/cmapss/test_all.parquet \
  --out /content/sequences/cmapss_test.pt \
  --sequence-length 24

In [ ]:
# 8. Train the shared temporal model
!PYTHONPATH=training/src python training/src/train.py \
  --data /content/sequences/cmapss_train.pt \
  --epochs 50 \
  --batch-size 128 \
  --lr 0.0003 \
  --out /content/artifacts/shared_temporal_v1.pt

In [ ]:
# 9. Evaluate against the official C-MAPSS test RUL targets
!PYTHONPATH=training/src python training/src/evaluate.py \
  --data /content/sequences/cmapss_test.pt \
  --model /content/artifacts/shared_temporal_v1.pt \
  --out /content/artifacts/cmapss_test_metrics.json

import json
print(json.dumps(json.load(open('/content/artifacts/cmapss_test_metrics.json')),indent=2))

## 10. Export the model bundle

The checkpoint travels with its configuration and evaluation metrics. This bootstrap checkpoint is **not** yet a validated MAINTAIN AI 24h/48h/7d failure predictor.

In [ ]:
import json, shutil
bundle=ART/'maintain_ai_shared_temporal_v1'
bundle.mkdir(exist_ok=True)
shutil.copy2(ART/'shared_temporal_v1.pt',bundle/'model.pt')
shutil.copy2(ART/'cmapss_test_metrics.json',bundle/'metrics.json')
shutil.copy2('training/config.yaml',bundle/'training_config.yaml')
metadata={
  'model_version':'shared-temporal-v1',
  'bootstrap_source':'nasa_cmapss_fd001_fd002_fd003_fd004',
  'sequence_length':24,
  'category_mapping':{'induction_motor':0,'pump':1,'compressor':2,'conveyor':3,'other':4},
  'note':'C-MAPSS RUL is source-native cycles; not MAINTAIN AI 24h/48h/7d failure risk.'
}
(bundle/'model_metadata.json').write_text(json.dumps(metadata,indent=2))
!cd /content && rm -f maintain_ai_shared_temporal_v1.zip && zip -qr maintain_ai_shared_temporal_v1.zip artifacts/maintain_ai_shared_temporal_v1
print('Bundle:',bundle)

In [ ]:
# 11. Download the exported bundle from Colab
from google.colab import files
files.download('/content/maintain_ai_shared_temporal_v1.zip')

## 12. What comes next

After this bootstrap run, add equipment-relevant public datasets and then MAINTAIN AI's own telemetry. The production model needs timestamp-aware 24h/48h/7d labels from real sensor histories plus technician-confirmed failures/maintenance outcomes. Do not manufacture months of MAINTAIN AI readings.